In [ ]:
import sys
sys.path.insert(0, "/synfs/resource/nb_resource/builtin")
print("path added")

In [ ]:
tenant_id: str = "acme"
batch_date: str = "2026-06-15"


In [ ]:
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "azure-ai-documentintelligence",
    "azure-keyvault-secrets",
    "azure-identity"
], check=True)
print("packages installed")

In [ ]:
# path configured in cell 0

In [ ]:
import sys
sys.path.insert(0, "/synfs/resource/nb_resource/builtin")

from dataclasses import asdict
from pyspark.sql.functions import current_timestamp, lit
from extractor.layout import _extract_sections as extract_sections
from extractor.splitter import SectionSplitter
from quality.suite import run_suite

print("imports successful")

In [ ]:
tenant_id = "acme"
batch_date = "2026-06-15"

DOCINTEL_ENDPOINT = "https://meridian-docintel.cognitiveservices.azure.com/"
DOCINTEL_KEY = "REPLACE_AT_RUNTIME"

BRONZE_PATH = f"/lakehouse/default/Files/bronze/{tenant_id}/{batch_date}"
SILVER_CHUNKS_TABLE = "silver_doc_chunks"
SILVER_REJECTED_TABLE = "silver_rejected"

print("tenant_id:", tenant_id)
print("batch_date:", batch_date)
print("DOCINTEL_ENDPOINT:", DOCINTEL_ENDPOINT)
print("DOCINTEL_KEY set:", bool(DOCINTEL_KEY))

In [ ]:
SUPPORTED_EXTENSIONS = (".pdf", ".docx", ".xlsx", ".pptx")

raw_files = [
    os.path.join(BRONZE_PATH, f)
    for f in os.listdir(BRONZE_PATH)
    if f.lower().endswith(SUPPORTED_EXTENSIONS)
]

if not raw_files:
    raise RuntimeError(f"no documents found in {BRONZE_PATH}")

print(f"{len(raw_files)} documents queued  tenant={tenant_id}  date={batch_date}")


In [ ]:
# extraction handled in cell 8

In [ ]:
import json

def chunk_to_dict(c, file_path):
    d = json.loads(json.dumps(asdict(c), default=str))
    d["section_count"] = int(d["section_count"])
    d["raw_path"] = file_path
    return d

all_dicts = []
for file_path in raw_files:
    doc_id = os.path.splitext(os.path.basename(file_path))[0]
    with open(file_path, "rb") as f:
        document_bytes = f.read()
    poller = client.begin_analyze_document(
        "prebuilt-layout",
        AnalyzeDocumentRequest(bytes_source=document_bytes),
    )
    result = poller.result()
    sections = _extract_sections(result)
    chunks = splitter.split(sections, doc_id=doc_id, tenant_id=tenant_id, doc_type="report")
    all_dicts.extend([chunk_to_dict(c, file_path) for c in chunks])

from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.types import IntegerType

batch_df = spark.createDataFrame(all_dicts)
batch_df = batch_df.withColumn("section_count", col("section_count").cast(IntegerType()))
batch_df = batch_df.withColumn("ingested_at", to_timestamp(col("ingested_at")))

pass_df, rejected_df = run_suite(batch_df)

pass_count = pass_df.count()
rejected_count = rejected_df.count()
total = pass_count + rejected_count

print(f"dq result  passed={pass_count}  rejected={rejected_count}  pass_rate={pass_count / max(total, 1):.1%}")

In [ ]:
(
    pass_df
    .withColumn("ingested_at", current_timestamp())
    .withColumn("batch_date", lit(batch_date))
    .write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(SILVER_CHUNKS_TABLE)
)

(
    rejected_df
    .withColumn("ingested_at", current_timestamp())
    .withColumn("batch_date", lit(batch_date))
    .write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(SILVER_REJECTED_TABLE)
)


In [ ]:
print("--- batch summary ---")
print(f"tenant:           {tenant_id}")
print(f"batch_date:       {batch_date}")
print(f"documents:        {len(raw_files)}")
print(f"chunks extracted: {len(all_chunks)}")
print(f"silver written:   {pass_count}")
print(f"rejected:         {rejected_count}")
print(f"pass rate:        {pass_count / max(total, 1):.1%}")
